# W14 · 02 SHAP 해석 · 부분의존도 · 특징 재정련

먼저 `01_features_xgb.ipynb` 를 실행해 `out/features.csv` 가 있어야 한다. 실행 = Claude Code 에 「`@notebooks/02_shap_refine.ipynb` 를 실행하고 결과를 요약해」(week14 폴더에서 `python ../run_nb.py notebooks/02_shap_refine.ipynb` · 직접 보려면 `jupyter lab`) → `out/shap_summary.json` · `out/pdp_density.json` · `out/refine_rounds.json`. 설명은 덱 17 ~ 25장.

In [ ]:
# 로컬 · Claude Code 가 기본: rdkit · xgboost · shap · scikit-learn 은 mdai 에 포함(../../environment.yml) · 설치 불필요
# (선택) Colab 이면 아래 줄의 주석을 푼다 (1회 · 약 1분)
# !pip -q install rdkit xgboost shap scikit-learn pandas
import subprocess, sys, json, os, pathlib
if pathlib.Path('scripts').is_dir() and pathlib.Path('notebooks').is_dir():
    os.chdir('notebooks')   # run_nb.py(week14 에서 시작) → 이 노트북의 경로 기준(notebooks/)으로 · JupyterLab 은 그대로
import pandas as pd
def run(script, *args):
    r = subprocess.run([sys.executable, f'scripts/{script}', *args], cwd='..', capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-800:])

## 1. TreeSHAP 4표적 · 워터폴 5행 · 부분의존도 (덱 17 ~ 19 · 23장)

In [ ]:
run('shap_report.py')
sh = json.load(open('../out/shap_summary.json'))
print(list(sh.keys()))
print(json.dumps(sh, ensure_ascii=False)[:1500])

In [ ]:
wf = json.load(open('../out/shap_waterfall.json'))
print(json.dumps(wf, ensure_ascii=False)[:1200])
pdp = json.load(open('../out/pdp_density.json'))
print({k: (v.get('span') if isinstance(v, dict) else None) for k, v in pdp.items()})

## 2. 재정련 R0 → R3 (덱 24 · 25장 · P14-3)

In [ ]:
run('refine_features.py')
rf = json.load(open('../out/refine_rounds.json'))
display(pd.DataFrame(rf['rounds']))
print('최종', rf['final'])
print('상관 쌍', rf['corr_pairs'])

**확인** 밀도 SHAP 1순위 온도 · 기준값 1.0108 · 재정련 17 → 6 · 밀도 0.9752 → 0.9764 (`../expected_output/` 과 비교)

**바꿔 보기** Claude Code 에 「`scripts/refine_features.py` 의 상위 k 만 6 → 4 로 바꾼 변형을 `out/` 아래 새 스크립트로 만들어 돌리고, 결과를 다른 이름으로 저장해 `out/refine_rounds.json` 과 비교해」 (`scripts/` 는 고치지 않는다). 사람의 판단: R2 가 남긴 logP(계산값)를 O 분율(원소분석)로 바꾸면 성능이 얼마나 변하는가.